# CARLA Binary Classifiers with LoRA Fine-Tuned ResNet-18

This notebook trains **three independent binary classifiers** on forward-facing CARLA camera images:
- 🚶 **Pedestrian Detector** (`has_pedestrian`)
- 🚦 **Traffic-Light Detector** (`has_traffic_light`)
- 🚗 **Vehicle Detector** (`has_vehicle`)

Each classifier is a **ResNet-18** backbone with **LoRA (Low-Rank Adaptation)** applied to its linear and convolutional layers so that only a small set of trainable parameters is added while the pre-trained weights stay frozen.

**How to use:**
1. Upload your dataset to Google Drive so the path `MyDrive/carnomaly/dataset/train/` contains the numbered images and `MyDrive/carnomaly/labels.csv` contains the label file.
2. Run all cells top-to-bottom in Google Colab (Runtime → Run all).
3. Trained weights are saved back to `MyDrive/carnomaly/weights/`.

**Runtime:** Set *Runtime → Change runtime type → GPU* (T4 or better) for fast training.

## 1 — Install dependencies

In [ ]:
# Install all required packages.
# loralib  → lightweight LoRA primitives for PyTorch
# safetensors → safe, fast weight serialization format
# scikit-learn → precision / recall / F1 helpers
%pip install -q loralib safetensors scikit-learn

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 — Imports & global settings

In [ ]:
import os
import copy
import math
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

import loralib as lora
from safetensors.torch import save_file, load_file

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 4 — Configuration

Edit the paths and hyper-parameters in this single cell.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/carnomaly'
IMAGES_DIR   = os.path.join(DRIVE_BASE, 'dataset', 'train')
LABELS_CSV   = os.path.join(DRIVE_BASE, 'labels.csv')
WEIGHTS_DIR  = os.path.join(DRIVE_BASE, 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# ── Dataset split ────────────────────────────────────────────────────────────
VAL_FRACTION  = 0.15   # 15 % of data used for validation
TEST_FRACTION = 0.10   # 10 % of data used for final test

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_EPOCHS  = 15
LR          = 3e-4    # learning rate for LoRA params + classifier head
WEIGHT_DECAY = 1e-4

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK    = 8      # rank of the low-rank decomposition
LORA_ALPHA   = 16     # scaling factor; effective lr ∝ alpha/rank
LORA_DROPOUT = 0.05

# ── Task names (column names in labels.csv) ───────────────────────────────────
TASKS = [
    'has_pedestrian',
    'has_traffic_light',
    'has_vehicle',
]

print('Configuration loaded.')
print(f'  Images : {IMAGES_DIR}')
print(f'  Labels : {LABELS_CSV}')
print(f'  Weights: {WEIGHTS_DIR}')

## 5 — Dataset

In [ ]:
# ── Image transforms ─────────────────────────────────────────────────────────
# ImageNet mean/std used because the ResNet-18 backbone was pre-trained on ImageNet.
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=_MEAN, std=_STD),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=_MEAN, std=_STD),
])


class CARLADataset(Dataset):
    """Dataset for CARLA forward-facing camera images.

    Args:
        images_dir  : directory containing the numbered frame images.
        labels_csv  : path to the CSV file with columns
                      frame, has_traffic_light, has_pedestrian, has_vehicle, …
        task        : one of 'has_pedestrian', 'has_traffic_light', 'has_vehicle'.
        transform   : torchvision transform to apply to each image.
    """

    _IMG_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.bmp']

    def __init__(self, images_dir: str, labels_csv: str, task: str,
                 transform=None):
        self.images_dir = images_dir
        self.task       = task
        self.transform  = transform

        df = pd.read_csv(labels_csv)
        # Normalize column names (strip whitespace)
        df.columns = df.columns.str.strip()

        required = {'frame', task}
        missing  = required - set(df.columns)
        if missing:
            raise ValueError(f'labels.csv is missing columns: {missing}')

        # Resolve image paths: the frame column may hold just the integer index
        # (e.g. 0, 1, 2) or a filename (e.g. "0.png").
        self.samples = []
        for _, row in df.iterrows():
            frame_val = str(row['frame']).strip()
            img_path  = self._resolve_path(frame_val)
            if img_path is None:
                continue   # silently skip missing images
            label = int(row[task])
            self.samples.append((img_path, label))

        if len(self.samples) == 0:
            raise RuntimeError(
                f'No valid samples found.  '
                f'Check that {images_dir} contains the frame images.'
            )

        pos = sum(l for _, l in self.samples)
        print(f'[{task}] {len(self.samples)} samples  '
              f'| pos={pos} ({100*pos/len(self.samples):.1f}%)  '
              f'| neg={len(self.samples)-pos}')

    def _resolve_path(self, frame_val: str):
        """Return the full image path for a frame identifier, or None."""
        # 1. Maybe frame_val is already a filename with extension.
        candidate = os.path.join(self.images_dir, frame_val)
        if os.path.isfile(candidate):
            return candidate
        # 2. Try appending common extensions.
        for ext in self._IMG_EXTENSIONS:
            candidate = os.path.join(self.images_dir, frame_val + ext)
            if os.path.isfile(candidate):
                return candidate
        return None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


def make_dataloaders(task: str):
    """Split the full dataset into train / val / test DataLoaders."""
    full_dataset = CARLADataset(
        IMAGES_DIR, LABELS_CSV, task, transform=eval_transform
    )
    n_total = len(full_dataset)
    n_test  = max(1, int(n_total * TEST_FRACTION))
    n_val   = max(1, int(n_total * VAL_FRACTION))
    n_train = n_total - n_val - n_test

    train_ds, val_ds, test_ds = random_split(
        full_dataset,
        [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(SEED),
    )

    # Apply augmentation only to the training subset
    train_ds.dataset = copy.copy(full_dataset)
    train_ds.dataset.transform = train_transform

    loaders = {
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True),
        'val'  : DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True),
        'test' : DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True),
    }
    print(f'  Split  → train={n_train}  val={n_val}  test={n_test}')
    return loaders


print('Dataset helpers defined.')

## 6 — LoRA-ResNet-18 Model

In [ ]:
def build_lora_resnet18(task_name: str,
                        lora_rank: int    = LORA_RANK,
                        lora_alpha: int   = LORA_ALPHA,
                        lora_dropout: float = LORA_DROPOUT) -> nn.Module:
    """Build a ResNet-18 binary classifier with LoRA adapters.

    Strategy
    --------
    * Load the official ImageNet-pre-trained ResNet-18 weights.
    * Freeze *all* original parameters.
    * Replace the final FC layer with a ``loralib.Linear`` layer
      (rank=lora_rank) that outputs 2 logits (binary CE loss).
    * Additionally add LoRA adapters to the last residual block's
      convolutional layers so the model can adapt image features
      beyond the head.
    * Only the LoRA delta-weight matrices (A, B) and the new head
      bias are trained; everything else stays frozen.

    Returns the model placed on DEVICE.
    """
    # ── 1. Load pre-trained backbone ────────────────────────────────────────
    backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    # ── 2. Freeze all existing parameters ───────────────────────────────────
    for param in backbone.parameters():
        param.requires_grad = False

    # ── 3. Add LoRA to the last residual block (layer4) ─────────────────────
    # We patch every nn.Conv2d inside layer4 with a lora.Conv2d that keeps
    # the original kernel but adds low-rank side-branches.
    for name, module in list(backbone.layer4.named_modules()):
        if isinstance(module, nn.Conv2d):
            # Walk the attribute path to replace the module in-place
            parts  = name.split('.')
            parent = backbone.layer4
            for part in parts[:-1]:
                parent = getattr(parent, part)
            old_conv = getattr(parent, parts[-1])
            new_conv = lora.Conv2d(
                in_channels  = old_conv.in_channels,
                out_channels = old_conv.out_channels,
                kernel_size  = old_conv.kernel_size[0],
                r            = lora_rank,
                lora_alpha   = lora_alpha,
                stride       = old_conv.stride[0],
                padding      = old_conv.padding[0] if (isinstance(old_conv.padding, tuple) and len(old_conv.padding) == 1) else (old_conv.padding if isinstance(old_conv.padding, int) else old_conv.padding[0]),
                bias         = old_conv.bias is not None,
            )
            # Copy pre-trained weights into the LoRA conv (frozen base)
            with torch.no_grad():
                new_conv.weight.copy_(old_conv.weight)
                if old_conv.bias is not None:
                    new_conv.bias.copy_(old_conv.bias)
            # Freeze the base weight; LoRA matrices remain trainable
            new_conv.weight.requires_grad = False
            if new_conv.bias is not None:
                new_conv.bias.requires_grad = False
            setattr(parent, parts[-1], new_conv)

    # ── 4. Replace the classifier head with a LoRA Linear layer ─────────────
    in_features = backbone.fc.in_features   # 512 for ResNet-18
    backbone.fc = lora.Linear(
        in_features  = in_features,
        out_features = 2,          # binary: class 0 or class 1
        r            = lora_rank,
        lora_alpha   = lora_alpha,
        lora_dropout = lora_dropout,
        merge_weights = False,     # keep A/B separate during training
    )

    # ── 5. Mark only LoRA parameters as trainable ────────────────────────────
    lora.mark_only_lora_as_trainable(backbone)

    # ── 6. Summary ───────────────────────────────────────────────────────────
    total_params    = sum(p.numel() for p in backbone.parameters())
    trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    print(f'[{task_name}] Model built.')
    print(f'  Total params    : {total_params:,}')
    print(f'  Trainable params: {trainable_params:,}  '
          f'({100*trainable_params/total_params:.2f}%)')

    return backbone.to(DEVICE)


print('Model builder defined.')

## 7 — Training & Evaluation Utilities

In [ ]:
# ── Class-balanced loss weight ────────────────────────────────────────────────
def compute_class_weights(loader: DataLoader) -> torch.Tensor:
    """Compute inverse-frequency weights for CE loss to handle class imbalance."""
    labels = torch.cat([y for _, y in loader]).numpy()
    counts  = np.bincount(labels, minlength=2).astype(float)
    weights = (counts.sum() / (2.0 * counts)).astype(np.float32)
    return torch.tensor(weights, device=DEVICE)


# ── One epoch of training ─────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


# ── Evaluation on val/test set ────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(images)
            loss   = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss  = running_loss / len(loader.dataset)
    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, acc, precision, recall, f1, all_labels, all_preds


# ── Full training loop for one task ──────────────────────────────────────────
def train_classifier(task: str):
    """Train a LoRA-ResNet-18 binary classifier for *task* and return the
    best model (by validation F1), the training history, and the test metrics.
    """
    print(f'\n{"="*60}')
    print(f'  Training classifier for: {task}')
    print(f'{"="*60}')

    # Data
    loaders = make_dataloaders(task)

    # Model
    model = build_lora_resnet18(task)

    # Class-balanced loss
    class_weights = compute_class_weights(loaders['train'])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Optimizer: only update LoRA parameters
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)

    # Cosine annealing scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    # Mixed-precision scaler
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

    history   = {'train_loss': [], 'train_acc': [], 'val_loss': [],
                 'val_acc': [], 'val_f1': []}
    best_f1   = -1.0
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_epoch(model, loaders['train'], criterion,
                                      optimizer, scaler)
        vl_loss, vl_acc, vl_prec, vl_rec, vl_f1, _, _ = evaluate(
            model, loaders['val'], criterion
        )
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['val_f1'].append(vl_f1)

        elapsed = time.time() - t0
        print(f'  Epoch {epoch:02d}/{NUM_EPOCHS}  '
              f'loss={tr_loss:.4f}  acc={tr_acc:.4f}  '
              f'| val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}  '
              f'val_F1={vl_f1:.4f}  '
              f'[{elapsed:.1f}s]')

        if vl_f1 > best_f1:
            best_f1   = vl_f1
            best_state = copy.deepcopy(model.state_dict())

    # Restore best checkpoint
    model.load_state_dict(best_state)

    # ── Final test evaluation ────────────────────────────────────────────────
    _, te_acc, te_prec, te_rec, te_f1, te_labels, te_preds = evaluate(
        model, loaders['test'], criterion
    )
    test_metrics = {
        'accuracy' : te_acc,
        'precision': te_prec,
        'recall'   : te_rec,
        'f1'       : te_f1,
        'labels'   : te_labels,
        'preds'    : te_preds,
    }

    print(f'\n  ── Test Results ({task}) ──')
    print(f'  Accuracy : {te_acc:.4f}')
    print(f'  Precision: {te_prec:.4f}')
    print(f'  Recall   : {te_rec:.4f}')
    print(f'  F1-Score : {te_f1:.4f}')

    return model, history, test_metrics


print('Training utilities defined.')

## 8 — Train all three classifiers

> ⏱ With a T4 GPU and ~1 000 images this typically takes **2–5 minutes** per classifier.

In [ ]:
results   = {}   # task → (model, history, test_metrics)

for task in TASKS:
    model, history, test_metrics = train_classifier(task)
    results[task] = (model, history, test_metrics)

print('\n✅  All three classifiers trained successfully!')

## 9 — Save weights to Google Drive

In [ ]:
for task, (model, _, _) in results.items():
    # Merge LoRA weights into the base weights before saving
    # (optional: creates a single stand-alone weight file)
    model.eval()

    # lora_state_dict() returns *only* the LoRA delta parameters (A, B matrices).
    # We save the *full* state dict so the model can be reloaded without the
    # loralib library if desired.
    full_state = model.state_dict()
    lora_only  = lora.lora_state_dict(model)

    # safetensors requires str keys and tensor values
    full_path = os.path.join(WEIGHTS_DIR, f'{task}_full.safetensors')
    lora_path = os.path.join(WEIGHTS_DIR, f'{task}_lora_only.safetensors')

    save_file(full_state, full_path)
    save_file(lora_only,  lora_path)

    print(f'Saved  {task}:')
    print(f'  Full  → {full_path}')
    print(f'  LoRA  → {lora_path}')

print('\n✅  All weights saved.')

## 10 — Evaluation Metrics & Visualisations

In [ ]:
# ── Metric summary table ──────────────────────────────────────────────────────
rows = []
for task, (_, _, metrics) in results.items():
    rows.append({
        'Task'     : task,
        'Accuracy' : f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall'   : f"{metrics['recall']:.4f}",
        'F1-Score' : f"{metrics['f1']:.4f}",
    })

summary_df = pd.DataFrame(rows).set_index('Task')
print('\n── Test-Set Evaluation Summary ─────────────────────────────────────')
print(summary_df.to_string())
summary_df

In [ ]:
# ── Per-task classification reports ──────────────────────────────────────────
for task, (_, _, metrics) in results.items():
    print(f'\n── {task} ─────────────────────────────────────────────────────────')
    print(classification_report(
        metrics['labels'], metrics['preds'],
        target_names=['absent (0)', 'present (1)'],
        zero_division=0,
    ))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')

for ax, task in zip(axes, TASKS):
    _, _, metrics = results[task]
    cm = confusion_matrix(metrics['labels'], metrics['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['absent', 'present'],
                yticklabels=['absent', 'present'])
    ax.set_title(task.replace('has_', '').replace('_', ' ').title())
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(WEIGHTS_DIR, 'confusion_matrices.png'), dpi=120)
plt.show()

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

epochs = range(1, NUM_EPOCHS + 1)
for row_idx, task in enumerate(TASKS):
    _, history, _ = results[task]
    short_name = task.replace('has_', '').replace('_', ' ').title()

    # Loss
    ax_loss = axes[row_idx, 0]
    ax_loss.plot(epochs, history['train_loss'], label='train')
    ax_loss.plot(epochs, history['val_loss'],   label='val')
    ax_loss.set_title(f'{short_name} — Loss')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('CE Loss')
    ax_loss.legend()
    ax_loss.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    # Accuracy / F1
    ax_acc = axes[row_idx, 1]
    ax_acc.plot(epochs, history['val_acc'], label='val acc')
    ax_acc.plot(epochs, history['val_f1'],  label='val F1')
    ax_acc.set_title(f'{short_name} — Val Accuracy & F1')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Score')
    ax_acc.set_ylim(0, 1.05)
    ax_acc.legend()
    ax_acc.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(os.path.join(WEIGHTS_DIR, 'training_curves.png'), dpi=120)
plt.show()

In [ ]:
# ── Metric bar chart ──────────────────────────────────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_keys  = ['accuracy', 'precision', 'recall', 'f1']
task_labels  = [t.replace('has_', '').replace('_', ' ').title() for t in TASKS]

x     = np.arange(len(metric_names))
width = 0.25
colours = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (task, colour) in enumerate(zip(TASKS, colours)):
    _, _, metrics = results[task]
    values = [metrics[k] for k in metric_keys]
    bars = ax.bar(x + i * width, values, width, label=task_labels[i],
                  color=colour, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Test-Set Metrics per Classifier')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(WEIGHTS_DIR, 'metric_comparison.png'), dpi=120)
plt.show()

## 11 — (Optional) Reload a saved model

Use this cell to reload a previously saved full model for inference without retraining.

In [ ]:
def load_classifier(task: str) -> nn.Module:
    """Rebuild the LoRA-ResNet-18 architecture and load saved weights."""
    model      = build_lora_resnet18(task)
    weight_path = os.path.join(WEIGHTS_DIR, f'{task}_full.safetensors')
    state_dict  = load_file(weight_path, device=str(DEVICE))
    model.load_state_dict(state_dict)
    model.eval()
    print(f'Loaded {task} from {weight_path}')
    return model

# Example usage (uncomment to run):
# ped_model = load_classifier('has_pedestrian')

## 12 — (Optional) Single-image inference demo

In [ ]:
@torch.no_grad()
def predict_image(model: nn.Module, image_path: str, task: str):
    """Run inference on a single image and print the result."""
    image  = Image.open(image_path).convert('RGB')
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
        logits = model(tensor)

    probs      = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    prediction = int(probs.argmax())
    confidence = float(probs[prediction])

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    ax.imshow(image)
    ax.axis('off')
    label_text = 'PRESENT' if prediction == 1 else 'ABSENT'
    ax.set_title(
        f'{task}\nPrediction: {label_text}  (confidence: {confidence:.1%})',
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

    return prediction, confidence

# Example usage (uncomment and set a real path):
# predict_image(results['has_pedestrian'][0],
#               '/content/drive/MyDrive/carnomaly/dataset/train/0.png',
#               'has_pedestrian')